In [1]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import minimize

In [11]:
def laplacian_fx(x, b = 3.0):
    return (1.0 / (2 * b)) * np.exp(-np.abs(x) / b)

def calculate_uniform_distortion(delta, b):
    y3 = delta/2.0
    integral_3 = lambda x: ((x - y3)**2) * laplacian_fx(x, b)
    d3, _ = quad(integral_3, 0, delta)

    y4 = 3.0 * delta / 2.0
    integral_4 = lambda x: ((x - y4)**2) * laplacian_fx(x, b)
    d4, _ = quad(integral_4, delta, np.inf)

    return 2 * (d3 + d4)

def optimize_uniform_quantizer(b, intital_delta = 4.0):
    def obj(d):
        return calculate_uniform_distortion(d[0], b)

    result = minimize(obj, [intital_delta], bounds=[(0.1, 20.0)])
    optimal_delta = result.x[0]

    yi = [-1.5*optimal_delta, -0.5*optimal_delta, 0.5*optimal_delta, 1.5*optimal_delta]
    bi = [-np.inf, -optimal_delta, 0, optimal_delta, np.inf]
    min_dist = calculate_uniform_distortion(optimal_delta, b)
    return optimal_delta, bi, yi, min_dist

def calculate_centroid(lower, upper, b):
    numerator_func = lambda x: x * laplacian_fx(x, b)
    denominator_func = lambda x: laplacian_fx(x, b)

    num, _ = quad(numerator_func, lower, upper)
    den, _ = quad(denominator_func, lower, upper)

    return num/den if den!= 0 else 0

def calculate_total_distortion(boundaries, points, b):
    dist = 0
    for i in range(len(points)):
        integral = lambda x: ((x - points[i]) ** 2) * laplacian_fx(x, b)
        val, _ = quad(integral, boundaries[i], boundaries[i+1])
        dist += val
    return dist

def optimize_lloyd_max(initial_points, b, tolerance = 1e-6):
    points = np.array(initial_points, dtype=float)
    boundaries = np.array([-np.inf, 0.0, 0.0, 0.0, np.inf], dtype=float)

    prev_dist = np.inf
    itr = 0

    while True:
        itr += 1
        for i in range(1, len(points)):
            boundaries[i] = (points[i-1] + points[i]) / 2.0
        
        for i in range(len(points)):
            points[i] = calculate_centroid(boundaries[i], boundaries[i+1], b)
        
        curr_dist = calculate_total_distortion(boundaries, points, b)
        if abs(prev_dist - curr_dist) < tolerance:
            break

        prev_dist = curr_dist
    
    return points.tolist(), boundaries.tolist(), curr_dist, itr

In [8]:
laplacian_b = 3.0

### Part 1: Uniform Quantizer

In [12]:
opt_uniform_delta, uniform_boundaries, uniform_points, uniform_dist = optimize_uniform_quantizer(laplacian_b)

In [34]:
print(f"Optimal Step Size (Delta) = {opt_uniform_delta:.4f}")
print(f"Decision Boundaries (b)   = {[float(round(val, 4)) if np.isfinite(val) else val for val in uniform_boundaries]}")
print(f"Reconstruction Points (y) = {[round(val, 4).item() for val in uniform_points]}")
print(f"Minimum Distortion (MSE)  = {uniform_dist:.4f}")

Optimal Step Size (Delta) = 4.6134
Decision Boundaries (b)   = [-inf, -4.6134, 0.0, 4.6134, inf]
Reconstruction Points (y) = [-6.9201, -2.3067, 2.3067, 6.9201]
Minimum Distortion (MSE)  = 3.5334


### Part 2: LLOYD-MAX Quantizer

In [35]:
lloyd_points, lloyd_boundaries, lloyd_dist, itr = optimize_lloyd_max(uniform_points, laplacian_b)

In [39]:
print(f"Converged in {itr} iterations.")
print(f"Decision Boundaries (b)   = {[round(val, 4) if np.isfinite(val) else val for val in lloyd_boundaries]}")
print(f"Reconstruction Points (y) = {[round(val, 4) for val in lloyd_points]}")
print(f"Minimum Distortion (MSE)  = {lloyd_dist:.4f}\n")

Converged in 11 iterations.
Decision Boundaries (b)   = [-inf, -4.7793, 0.0, 4.7793, inf]
Reconstruction Points (y) = [-7.7793, -1.7805, 1.7805, 7.7793]
Minimum Distortion (MSE)  = 3.1715



In [40]:
print(f"Uniform MSE   : {uniform_dist:.4f}")
print(f"Lloyd-Max MSE : {lloyd_dist:.4f}")
print(f"Improvement   : {uniform_dist - lloyd_dist:.4f} reduction in MSE.")

Uniform MSE   : 3.5334
Lloyd-Max MSE : 3.1715
Improvement   : 0.3619 reduction in MSE.
